### Rag PipeLines - Data Ignestion to Vector DB Pipeline

In [1]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\PMLS\AppData\Local\Temp\ipykernel_286228\3728471183.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
d:\usama\Ai Engineering\Rag\rag_by_krish\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
### Read all pdf inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory recursively."""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    if not pdf_dir.exists():
        raise FileNotFoundError(f"Directory not found: {pdf_directory}")

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    if not pdf_files:
        print("No PDF files found.")
        return []

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["source_path"] = str(pdf_file)
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(f"✔ Loaded {len(documents)} pages.")

        except Exception as e:
            print(f"❌ Error processing {pdf_file.name}: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents

In [20]:
# Process all pdfs in a directory 
all_pdf_documents = process_all_pdfs("../data/pdf")

Found 4 PDF files to process.

Processing: ..\data\pdf\9-cloud-computing.pdf
✔ Loaded 54 pages.

Processing: ..\data\pdf\aiResume.pdf
✔ Loaded 1 pages.

Processing: ..\data\pdf\printData (1) (1).pdf
✔ Loaded 2 pages.

Processing: ..\data\pdf\Resume-for HR.pdf
✔ Loaded 1 pages.

Total documents loaded: 58


In [21]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2015-05-07T08:51:56+02:00', 'title': 'Slide 1', 'author': 'Jim Dowling', 'moddate': '2015-05-07T08:51:56+02:00', 'source': '..\\data\\pdf\\9-cloud-computing.pdf', 'total_pages': 54, 'page': 0, 'page_label': '1', 'source_file': '9-cloud-computing.pdf', 'source_path': '..\\data\\pdf\\9-cloud-computing.pdf', 'file_type': 'pdf'}, page_content='ID2210\nJim Dowling\nIntroduction to Cloud Computing'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2015-05-07T08:51:56+02:00', 'title': 'Slide 1', 'author': 'Jim Dowling', 'moddate': '2015-05-07T08:51:56+02:00', 'source': '..\\data\\pdf\\9-cloud-computing.pdf', 'total_pages': 54, 'page': 1, 'page_label': '2', 'source_file': '9-cloud-computing.pdf', 'source_path': '..\\data\\pdf\\9-cloud-computing.pdf', 'file_type': 'pdf'}, page_content='Cloud Computing\

In [22]:
def split_documents(documents, chunk_size = 1000 , chunk_overlap = 200):
    """Split documents into smaller chunks for better rag performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n" , " " , ""]

    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} document into {len(split_docs)} chunks")

    if split_docs:
        print(f"\n example chunks")
        print(f"Content : {split_docs[0].page_content[:200]}...")
        print(f"Meta data {split_docs[0].metadata}")

    return split_docs

In [23]:
chunks = split_documents(all_pdf_documents)
chunks

Split 58 document into 64 chunks

 example chunks
Content : ID2210
Jim Dowling
Introduction to Cloud Computing...
Meta data {'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2015-05-07T08:51:56+02:00', 'title': 'Slide 1', 'author': 'Jim Dowling', 'moddate': '2015-05-07T08:51:56+02:00', 'source': '..\\data\\pdf\\9-cloud-computing.pdf', 'total_pages': 54, 'page': 0, 'page_label': '1', 'source_file': '9-cloud-computing.pdf', 'source_path': '..\\data\\pdf\\9-cloud-computing.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2015-05-07T08:51:56+02:00', 'title': 'Slide 1', 'author': 'Jim Dowling', 'moddate': '2015-05-07T08:51:56+02:00', 'source': '..\\data\\pdf\\9-cloud-computing.pdf', 'total_pages': 54, 'page': 0, 'page_label': '1', 'source_file': '9-cloud-computing.pdf', 'source_path': '..\\data\\pdf\\9-cloud-computing.pdf', 'file_type': 'pdf'}, page_content='ID2210\nJim Dowling\nIntroduction to Cloud Computing'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2015-05-07T08:51:56+02:00', 'title': 'Slide 1', 'author': 'Jim Dowling', 'moddate': '2015-05-07T08:51:56+02:00', 'source': '..\\data\\pdf\\9-cloud-computing.pdf', 'total_pages': 54, 'page': 1, 'page_label': '2', 'source_file': '9-cloud-computing.pdf', 'source_path': '..\\data\\pdf\\9-cloud-computing.pdf', 'file_type': 'pdf'}, page_content='Cloud Computing\

### Embedding and vectorStore DB

In [24]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings 
import uuid
from typing import List , Dict , Any , Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [25]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformers."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding model.

        Args:
            model_name (str): Hugging Face Sentence Transformer model name.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")

            self.model = SentenceTransformer(self.model_name)

            dimension = self.model.get_sentence_embedding_dimension()
            print(f"Model loaded successfully. Embedding dimension: {dimension}")

        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise

    def generate_embeddings(self , texts:List[str]) -> np.ndarray:
        """
            Generate embeddings for a list of texts.

            Args:
                texts: List of text strings to be embedded.

            Returns:
                NumPy array of embeddings with shape
                (len(texts), embedding_dimension).
        """

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings= self.model.encode(texts , show_progress_bar=True)
        print(f"Generate Embedding with shape: {embeddings.shape}")
        return embeddings

    def get_embedding_dimension(self) -> int:
        """Get the embedding dimension of the model"""
        if not self.model:
            raise ValueError("model not loaded")
        return self.model.get_sentence_embedding_dimension()


embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7864.65it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\PMLS\AppData\Local\Temp\ipykernel_286228\382511177.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.model.get_sentence_embedding_dimension()


### VectorStore

In [26]:
class Vectorstore:
    """Manages document embeddings in a chromaDb vectorStore"""

    def __init__(self , collection_name:str = "pdf_documents" , persist_directory: str = "../data/vector_store"):
        """
            Initialize the vector store

            Args:
                collection_name: Name of the ChromaDb collection
                persist_directory: Directory to persist the vector store 
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize chromaDB client and collection"""
        try:
            # create a persistent chromaDB client 
            os.makedirs(self.persist_directory , exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"Pdf documet embedding for rag"}
            )
            print(f"Vector Store initialized. Collection : {self.collection_name}")
            print(f"existing documets in collection : {self.collection.count()}")
        
        except Exception as e:
            print(f"Error initialize vector store : {e}")
            raise
    

    def add_documents(self , documents:List[Any] , embeddings:np.ndarray):
        """
            Add documents and  there embeddings to the vector store

            Args:
                documents : List of langchain documents 
                embeddings : Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match the number of embeddings")
        
        print(f"Adding {len(documents)} documents to the vector store")

        #prepare data for chroma db
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i , (doc,embedding) in enumerate(zip(documents,embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # document content 
            documents_text.append(doc.page_content)

            #embeddings
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector Db")
            print(f"total documents in collection : {self.collection.count()}")
        
        except Exception as e:
            print(f"Error adding documents to vector store : {e}")
            raise

vectorStore = Vectorstore()
vectorStore

Vector Store initialized. Collection : pdf_documents
existing documets in collection : 30


In [27]:
### Convert the text to embedding
texts=[doc.page_content for doc in chunks]

### Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

### store in the vector database 
vectorStore.add_documents(chunks,embeddings)


Generating embeddings for 64 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.22s/it]

Generate Embedding with shape: (64, 384)
Adding 64 documents to the vector store
Successfully added 64 documents to vector Db
total documents in collection : 94


### Retriver Pipeline from Vectorstore

In [28]:
class RAGRetriver:
    """Handles query based retrival from the vector store"""

    def __init__(self,vector_store:Vectorstore , embedding_manager:EmbeddingManager):
        """
            Initialize the retriver 

            Args : 
                vector_store : Vector store containing document embeddings
                embedding_manager : Manager for generating query embeddings 
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager


    def retrive(self , query : str, top_k:int=5,score_threshold:float = 0.0)->List[Dict[str,any]]:

        """
            Retrive the relevant document for a query

            Args : 
                query : the search query 
                top_k: NUmbers of top result to find 
                score_threshold: Minimum similarity score threshold 
            
            Returns: 
                List of dictioneries containing the retrived document and metadata
        """

        print(f"Retriving document for query : '{query}'");
        print(f"Top k: {top_k} , score threshold score threshold ");

        query_embedding = self.embedding_manager.generate_embeddings([query])[0];

        # search in Vrectore 
        try:
            results = self.vector_store.collection.query(
                query_embeddings= [query_embedding.tolist()],
                n_results=top_k
            )

            retrived_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id , document , metadata , distance) in enumerate(zip(ids, documents , metadatas , distances)):
                    # convert distance to similarity score (chromadb uses cosine distances)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrived_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })

                print(f"Retrived {len(retrived_docs)} documents (after filtering)")

            else:
                print(f"No document found")

            return retrived_docs
        
        except Exception as e:
            print(f"Error druing retrival : {e}")
            return []
        

rag_retriver = RAGRetriver(vectorStore , embedding_manager)

In [29]:
rag_retriver

In [31]:
rag_retriver.retrive("what is Community cloud")

Retriving document for query : 'what is Community cloud'
Top k: 5 , score threshold score threshold 
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.66it/s]

Generate Embedding with shape: (1, 384)
Retrived 5 documents (after filtering)


[{'id': 'doc_5c209acf_16',
  'content': 'Other types of Clouds\n•Community cloud\n- The cloud infrastructure is shared by several organizations \nand supports a specific community that has shared \nconcerns (e.g., mission, security requirements, policy, and \ncompliance considerations). \n•Hybrid cloud\n- The cloud infrastructure is a composition of two or more \nclouds (private, community, or public) that remain unique \nentities but are bound together by standardized or \nproprietary technology that enables data and application \nportability.',
  'metadata': {'producer': 'Microsoft® PowerPoint® 2013',
   'source_file': '9-cloud-computing.pdf',
   'page_label': '17',
   'total_pages': 54,
   'creator': 'Microsoft® PowerPoint® 2013',
   'page': 16,
   'creationdate': '2015-05-07T08:51:56+02:00',
   'doc_index': 16,
   'content_length': 498,
   'author': 'Jim Dowling',
   'file_type': 'pdf',
   'source': '..\\data\\pdf\\9-cloud-computing.pdf',
   'source_path': '..\\data\\pdf\\9-cloud-c